# Snowflake Horizon Catalog — Iceberg Interoperability Demo Notebook
# Import into Snowflake: Snowsight → Projects → Notebooks → Import
#
# Demonstrates all 9 Horizon Catalog capabilities:
#   1. Open Iceberg REST Access
#   2. Apache Polaris Integration
#   3. Single Endpoint
#   4. Read + Write from External Engines (PyIceberg + DuckDB)
#   5. Existing Snowflake Security Model
#   6. Credential Vending
#   7. Governed Multi-Engine Access
#   8. Policy Enforcement on Iceberg
#   9. Supported External Engines
#
# Account  : SCB47336 (vmedidademo-aws1)
# Endpoint : https://scb47336.snowflakecomputing.com/polaris/api/catalog
# Database : horizon_demo_db
# Table    : public.transactions

In [ ]:
import os, subprocess, sys

def pip(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

pip("pyiceberg[rest,pyarrow]", "duckdb", "snowflake-connector-python", "pandas")

# ── CONFIGURE THESE FOR YOUR ACCOUNT ─────────────────────────────────────────
SNOWFLAKE_ACCOUNT = os.getenv("SNOWFLAKE_ACCOUNT", "<your_account_identifier>")
SNOWFLAKE_USER    = os.getenv("SNOWFLAKE_USER",    "<your_user>")
WAREHOUSE_DB      = os.getenv("HORIZON_WAREHOUSE",  "horizon_demo_db")

HORIZON_URI = f"https://{SNOWFLAKE_ACCOUNT}.snowflakecomputing.com/polaris/api/catalog"
TOKEN       = os.getenv("SNOWFLAKE_TOKEN")  # Snowflake session token (from key-pair JWT or externalbrowser)

print(f"Horizon endpoint: {HORIZON_URI}")
if not TOKEN:
    print("WARNING: Set SNOWFLAKE_TOKEN env var with a valid session token to run REST cells.")
else:
    print("Token found. Ready to connect.")

## Feature 1 + 3: Open Iceberg REST Access via the Single Horizon Endpoint
Connect PyIceberg to the Horizon REST catalog.

In [ ]:
from pyiceberg.catalog.rest import RestCatalog

catalog = RestCatalog(
    name="horizon",
    **{
        "uri":      HORIZON_URI,
        "token":    TOKEN,
        "warehouse": WAREHOUSE,
        "header.X-Iceberg-Access-Delegation": "vended-credentials",
    },
)

print("=== Namespaces (Snowflake schemas) ===")
for ns in catalog.list_namespaces():
    print(" ", ns)

print("\n=== Tables in public namespace ===")
for tbl in catalog.list_tables(("public",)):
    print(" ", tbl)

## Feature 4: Read and Write from External Engines (PyIceberg)

In [ ]:
import pyarrow as pa
from pyiceberg.expressions import EqualTo, GreaterThan

table = catalog.load_table("public.transactions")
print("Schema:", table.schema())

print("\n=== Full scan ===")
df = table.scan().to_pandas()
display(df)

print("\n=== Filter: amount > 500 ===")
display(table.scan(row_filter=GreaterThan("amount", 500)).to_pandas())

print("\n=== Append two new rows ===")
new_rows = pa.table({
    "transaction_id": ["txn-008", "txn-009"],
    "customer_id":    ["cust-F",  "cust-G"],
    "amount":         [1100.00,    220.50],
    "currency":       ["USD",      "GBP"],
    "transaction_ts": pa.array(
        [1705392000000000, 1705395600000000], type=pa.timestamp("us")
    ),
    "status":         ["COMPLETED", "PENDING"],
    "region":         ["us-east",   "eu-west"],
})
table.append(new_rows)
print(f"Row count after append: {table.scan().to_pandas().shape[0]}")

## Feature 6: Credential Vending — Inspect vended S3 credentials

In [ ]:
print("=== Table location (Snowflake-managed S3) ===")
print(table.location())

print("\n=== Vended credentials (keys truncated for safety) ===")
for k, v in table.io.properties.items():
    if k.startswith("s3."):
        display_v = v[:10] + "..." if len(v) > 10 else v
        print(f"  {k}: {display_v}")

## Feature 9: DuckDB reads the same table through the same Horizon endpoint

In [ ]:
import duckdb

conn = duckdb.connect()
conn.execute("INSTALL iceberg; LOAD iceberg;")
conn.execute(f"""
    CREATE OR REPLACE SECRET hs (
        TYPE ICEBERG_REST, TOKEN '{TOKEN}',
        ENDPOINT '{HORIZON_URI}', WAREHOUSE '{WAREHOUSE}'
    );
""")
conn.execute(f"ATTACH '{HORIZON_URI}' AS h (TYPE ICEBERG_REST, SECRET 'hs');")

print("=== DuckDB: full scan of transactions via Horizon ===")
display(conn.execute("SELECT * FROM h.public.transactions ORDER BY transaction_ts").fetchdf())

print("\n=== DuckDB: aggregation by currency ===")
display(conn.execute("""
    SELECT currency, COUNT(*) AS txn_count, ROUND(SUM(amount), 2) AS total
    FROM h.public.transactions GROUP BY currency ORDER BY total DESC
""").fetchdf())

conn.close()

## Feature 7: Governed Multi-Engine Consistency
Snowflake native SQL vs PyIceberg: same query, same result

In [ ]:
import snowflake.connector, pandas as pd

sf_conn = snowflake.connector.connect(
    account      = "scb47336",
    user         = "VMEDIDA",
    role         = "ACCOUNTADMIN",
    warehouse    = "COMPUTE_WH",
    database     = "horizon_demo_db",
    schema       = "public",
    authenticator = "externalbrowser",
)

SF_QUERY = """
    SELECT currency, COUNT(*) AS txn_count, ROUND(SUM(amount), 2) AS total
    FROM transactions WHERE status = 'COMPLETED'
    GROUP BY currency ORDER BY total DESC
"""
sf_df = pd.read_sql(SF_QUERY, sf_conn)
sf_conn.close()

PYICE_QUERY = """
    SELECT currency, COUNT(*) AS txn_count, ROUND(SUM(amount), 2) AS total
    FROM h.public.transactions WHERE status = 'COMPLETED'
    GROUP BY currency ORDER BY total DESC
"""
conn = duckdb.connect()
conn.execute("LOAD iceberg;")
conn.execute(f"""
    CREATE OR REPLACE SECRET hs (
        TYPE ICEBERG_REST, TOKEN '{TOKEN}',
        ENDPOINT '{HORIZON_URI}', WAREHOUSE '{WAREHOUSE}'
    );
""")
conn.execute(f"ATTACH '{HORIZON_URI}' AS h (TYPE ICEBERG_REST, SECRET 'hs');")
duck_df = conn.execute(PYICE_QUERY).fetchdf()
conn.close()

print("Snowflake native:"); display(sf_df)
print("\nDuckDB/Horizon:");  display(duck_df)

## Feature 8: Policy Enforcement — verify via Snowflake Python connector

In [ ]:
import snowflake.connector, pandas as pd

def sf_query(role, query):
    conn = snowflake.connector.connect(
        account      = "scb47336",
        user         = "VMEDIDA",
        role         = role,
        warehouse    = "COMPUTE_WH",
        database     = "horizon_demo_db",
        schema       = "public",
        authenticator = "externalbrowser",
    )
    df = pd.read_sql(query, conn)
    conn.close()
    return df

Q = "SELECT transaction_id, customer_id, amount, region FROM transactions"

print("=== ACCOUNTADMIN: all rows, plain values ===")
display(sf_query("ACCOUNTADMIN", Q))

print("\n=== analyst_us: us-* rows only, masked values ===")
try:
    display(sf_query("analyst_us", Q))
except Exception as e:
    print(f"Role analyst_us error (expected if not created): {e}")

## Feature 5: Security Model — confirm grants via Snowflake Python connector

In [ ]:
conn = snowflake.connector.connect(
    account      = "scb47336",
    user         = "VMEDIDA",
    role         = "ACCOUNTADMIN",
    warehouse    = "COMPUTE_WH",
    authenticator = "externalbrowser",
)

print("=== Grants on transactions table ===")
display(pd.read_sql("SHOW GRANTS ON TABLE horizon_demo_db.public.transactions", conn))

print("\n=== Iceberg table info ===")
display(pd.read_sql(
    "SELECT SYSTEM\$GET_ICEBERG_TABLE_INFORMATION('horizon_demo_db.public.transactions') AS info",
    conn
))

print("\n=== Horizon endpoint ===")
display(pd.read_sql("SELECT SYSTEM\$GET_ICEBERG_REST_CATALOG_ENDPOINT() AS endpoint", conn))
conn.close()